# Domain Taxonomy via Gemini\n\nFeed the SR subdomain structure (2-digit prefix + sample law titles) to Gemini.\nAsk it to produce a clean taxonomy with descriptions suitable for query classification.

In [ ]:
import pandas as pd
import numpy as np

laws = pd.read_csv('../data/laws_de.csv')
laws['law_id'] = laws['citation'].str.extract(r'(\S+)$')

# Get the first Art. 1 paragraph per law (purpose statement)
art1 = laws[laws['citation'].str.match(r'Art\. 1 (Abs\. 1 )?\S+')].copy()
art1 = art1.drop_duplicates('law_id')

# Also pull the law's full name from the title prefix
art1['full_name'] = art1['title'].str.split(' - ').str[0].str.strip()

print(f'Laws with Art. 1: {len(art1)} / {laws["law_id"].nunique()}')

In [ ]:
# Sample 50 diverse laws — spread across named (USG, ZGB) and numeric SR (172.xxx)
art1['is_numeric'] = art1['law_id'].str.match(r'^\d')
named   = art1[~art1['is_numeric']]
numeric = art1[art1['is_numeric']]

sample = pd.concat([
    named.sample(30, random_state=42),
    numeric.sample(20, random_state=42)
]).reset_index(drop=True)

print(f'Sample size: {len(sample)}')
sample[['law_id', 'full_name', 'text']].head(10)

In [ ]:
# Build the prompt — show law_id + full_name + Art.1 text for each sample
lines = []
for _, row in sample.iterrows():
    lines.append(f'[{row["law_id"]}] {row["full_name"]}\n  Art.1: {str(row["text"])[:250]}')

law_block = '\n\n'.join(lines)

PROMPT = f"""You are a Swiss law expert. Below are 50 Swiss federal laws/ordinances with their Art. 1 purpose statements.

Your task:
1. Read all 50 entries.
2. Propose a taxonomy of legal domains that covers all of them.
   - Each domain should have a short German name (e.g. "Umweltrecht", "Sozialversicherungsrecht")
   - Domains should be at the right granularity: broad enough to be useful (not 200 single-law domains),
     specific enough to narrow retrieval (not just "Öffentliches Recht" for everything).
   - Aim for 20-40 domains total.
3. Assign each of the 50 laws to one or more domains.
4. Output format:
   DOMAINS:
   - <domain_id>: <German name> — <1-line description in English>
   ...

   ASSIGNMENTS:
   <law_id>: <domain_id1>, <domain_id2>
   ...

Laws:
{law_block}
"""

print(PROMPT[:500], '...')
print(f'\nTotal prompt length: {len(PROMPT)} chars')

In [ ]:
# Call Claude API (or swap for Gemini/OpenAI)
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

response = client.messages.create(
    model='claude-opus-4-6',
    max_tokens=4096,
    messages=[{'role': 'user', 'content': PROMPT}]
)

output = response.content[0].text
print(output)

In [ ]:
# After reviewing the output above, we'll parse domains and assignments
# Then scale to all 2048 laws in the next step

# Quick parse to inspect
lines_out = output.split('\n')
in_domains = False
in_assignments = False
domains = {}
assignments = {}

for line in lines_out:
    line = line.strip()
    if line.startswith('DOMAINS:'): in_domains = True; in_assignments = False; continue
    if line.startswith('ASSIGNMENTS:'): in_assignments = True; in_domains = False; continue
    if in_domains and line.startswith('- '):
        parts = line[2:].split(':', 1)
        if len(parts) == 2:
            domains[parts[0].strip()] = parts[1].strip()
    if in_assignments and ':' in line and not line.startswith('-'):
        law, doms = line.split(':', 1)
        assignments[law.strip()] = [d.strip() for d in doms.split(',')]

print(f'Domains found: {len(domains)}')
for k, v in domains.items():
    print(f'  {k}: {v}')

print(f'\nAssignments: {len(assignments)} laws assigned')